# 10 — Network statistics for Task B: Hypertension (issues #16, #15)

**The task:** whether a patient will receive a first hypertension diagnosis within the next year, predicted at the end of the discharge day. Each node is one labeled prediction (3,511 discharges from 2,217 patients). Base rate of the outcome: about 13%.

This notebook describes the three patient-similarity graphs for this task:

| Graph | Built from | Role |
|---|---|---|
| **safe** | Each node's diagnoses *before* the prediction time | The honest graph we'd actually use |
| **leaky** | The patient's diagnoses over their *whole* record, including the future | A deliberate cheat, used as a control |
| **null** | The safe graph with its edges randomly rewired. Every node keeps its exact degree, and the edge weights are shuffled | A "structure means nothing" control |

It answers four questions:

1. **Shape (#11).** How many connections do nodes have, how clustered is the graph, and which nodes sit in the middle?
2. **Groups (#12).** Do Louvain and Leiden agree on the groups? Do groups differ in their new hypertension rate?
3. **Alignment with the outcome (#13).** Homophily, and Dirichlet energy of the CLMBR fingerprints.
4. **What leakage actually changes (#15).** Which links the leaky graph adds or removes, how much of each graph is a patient linked to their own other admissions, and whether any single diagnosis code gives the outcome away.

**One null graph is not enough.** Every null number is the mean over **10 differently seeded null graphs**, with the range covering the middle 95% of them.

**Data (team decision, Oct 2 2026).** Everything here runs on the **course's OMOP tables**, which every team member has:
- **Diagnoses:** the tables' standard SNOMED concepts. ICD-9 and ICD-10 codes for one disease are already one concept, so there is no coding-era split (notebook 08).
- **Outcome labels:** rebuilt from the tables with EHRSHOT's published definitions (`ehrgraph.data.omop_labels`). They match Stanford's official labels on 99.1% (long stay) and 93% (hypertension) of prediction times, with 99.84% and 99.4% outcome agreement (notebook 11).
- **CLMBR fingerprints:** used only for Dirichlet energy, and only if *your own* copy of Stanford's full release is on this machine. Otherwise that one measure is skipped. Fingerprints are patient-level and are never shared between team members.

**Data rule.** This runs locally against on-device data. Only per-graph summary numbers (`results/aggregate/network_stats_task_b.csv`) and figures built from them (`results/aggregate/figures/10_*.png`) are committed. Per-node tables stay in memory.

**One code path.** All computation lives in `ehrgraph.netstats.task_analysis` and all figures in `ehrgraph.viz.network_figures`, shared by both tasks so their numbers are directly comparable (issue #17).

In [ ]:
# Bootstrap (see notebooks/README.md): find the repo root, then locate the
# on-device data -- course OMOP tables required, full release optional.
import sys
from pathlib import Path

_repo_root = Path.cwd()
while not (_repo_root / "pyproject.toml").exists() and _repo_root != _repo_root.parent:
    _repo_root = _repo_root.parent
sys.path.insert(0, str(_repo_root))

from scripts.local_setup import ensure_data_sources  # noqa: E402

sources = ensure_data_sources()

In [ ]:
import time

import matplotlib.pyplot as plt
import pandas as pd

from ehrgraph.data.omop import OmopDatabase, OmopTables, concept_names
from ehrgraph.data.omop_labels import rebuild_task_labels
from ehrgraph.eval.aggregate import write_aggregate
from ehrgraph.graphs.similarity import graph_summary
from ehrgraph.netstats.task_analysis import (
    PRIMARY_VARIANT,
    analyze_task,
    summary_table,
    task_graphs_from_labels,
)
from ehrgraph.viz import network_figures as nf

TASK = "new_hypertension"
POSITIVE, NEGATIVE = "new hypertension", "no new hypertension"
FIG = "10"  # figure-file prefix
results_dir = _repo_root / "results" / "aggregate"
figures_dir = results_dir / "figures"
nf.apply_style()

## Build the graphs and run the analysis

Four steps:

1. `OmopDatabase` presents the course tables in the same shape as the database the rest of the code reads, so the honest/cheating time cut is still the single tested function in `ehrgraph.data.loader`.
2. `rebuild_task_labels` creates the outcome labels.
3. `task_graphs_from_labels` builds the safe and leaky node vectors (diagnoses only, yes/no per node), turns each into a mutual-10-NN similarity graph, and makes 10 degree-preserving null graphs from the safe one. If a full-release copy is present, it also attaches CLMBR fingerprints where Stanford computed them.
4. `analyze_task` computes every metric in this notebook in one pass.

In [ ]:
t0 = time.time()
tables = OmopTables(sources.trimmed)
db = OmopDatabase(tables)  # condition events, SNOMED concepts
labels = rebuild_task_labels(tables, TASK)
tg = task_graphs_from_labels(db, labels, TASK, clmbr_path=sources.clmbr_features)
print(f"{len(tg.nodes):,} nodes, prevalence {tg.prevalence:.3f}; built in {time.time() - t0:.0f}s")
print(f"CLMBR fingerprints for {len(tg.feature_nodes):,} of {len(tg.nodes):,} nodes")

t0 = time.time()
an = analyze_task(tg, graph_variant=PRIMARY_VARIANT)
print(f"analysis: {len(an.rows)} aggregate rows in {time.time() - t0:.0f}s")
pd.DataFrame({name: graph_summary(g) for name, g in tg.graphs.items()}).T

---
# Part 1 — Shape of the graphs (issue #11)

- **Density**: the share of all possible node pairs that are connected.
- **Clustering** and **transitivity**: when two nodes share a neighbor, how often they are also connected (triadic closure, Lecture 3).
- **Degree assortativity**: whether well-connected nodes link to other well-connected nodes.
- **Path length** and **diameter**: hops between nodes in the largest connected piece. The average path is estimated from 500 sampled start nodes, which matched the exact value to 0.001 hops on Task A.

The null keeps every node's degree, so density and degree match the safe graph by construction. Its clustering and path lengths show how much structure exists beyond "who has how many links."

In [ ]:
STRUCT = [
    "density",
    "mean_degree",
    "max_degree",
    "avg_clustering",
    "transitivity",
    "degree_assortativity",
    "lcc_avg_shortest_path_hops",
    "lcc_diameter_hops_lower_bound",
    "n_components",
    "isolated_fraction",
    "largest_cc_fraction",
]
summary_table(an.rows, STRUCT)

### Degree distribution

Mutual k-NN with k = 10 caps every node at 10 links, so there are no hubs by design. The interesting part is the left side: a node with 0 or 1 links is one that nobody else counts among their closest matches.

In [ ]:
fig = nf.degree_figure(tg)
nf.save(fig, figures_dir / f"{FIG}_degree_distribution.png")
plt.show()

### Centrality

Betweenness and closeness use `distance = 1 − similarity`, and are computed exactly with igraph. Closeness and eigenvector centrality are computed inside the largest connected piece only. Eigenvector centrality *localizes* in graphs this modular: nearly all its weight lands on one dense cluster, so it is not a useful ranking of nodes.

Each AUROC below treats centrality alone as a predictor of new hypertension. 0.5 means no information. It is descriptive only: it uses every node's label.

In [ ]:
fig = nf.centrality_figure(an)
nf.save(fig, figures_dir / f"{FIG}_centrality_distributions.png")
plt.show()
summary_table(
    an.rows,
    [
        f"{m}_centrality_auroc_vs_label"
        for m in ("degree", "betweenness", "closeness", "eigenvector")
    ],
    conditions=("safe", "leaky"),
).round(3)

---
# Part 2 — Groups (issue #12)

Louvain and Leiden both maximize *modularity*: dense inside groups, sparse between them. Their agreement (adjusted Rand index, ARI) runs from 0 (chance) to 1 (identical).

The figure shows each Leiden group's new hypertension rate, for groups of at least 20 nodes. Smaller groups are noisy, and a tiny group's rate says too much about the few patients in it. **η²** is the share of outcome variation explained by group membership.

In [ ]:
summary_table(
    an.rows,
    [
        "adjusted_rand_index",
        "n_leiden_communities",
        "louvain_modularity",
        "leiden_modularity",
        "leiden_outcome_eta_squared",
        "leiden_outcome_nmi",
        "leiden_outcome_share_in_reported_communities",
    ],
)

In [ ]:
fig = nf.community_outcome_figure(tg, an, POSITIVE)
nf.save(fig, figures_dir / f"{FIG}_community_outcome_rates.png")
plt.show()

### What the graphs look like

The largest connected piece of each graph, drawn with a force-directed layout. Each dot is one node, colored only by outcome: **violet** = new hypertension, **light gray** = no new hypertension. Nothing is labeled or identifiable. Homophily shows up as violet dots clumping together; the null panel shows what no clumping looks like.

In [ ]:
fig = nf.layout_figure(tg, POSITIVE, NEGATIVE)
nf.save(
    fig, figures_dir / f"{FIG}_graph_layouts.png", dpi=80
)  # thousands of dots; keeps it < 500 KB
plt.show()

---
# Part 3 — Do the graphs line up with the outcome? (issue #13)

**Class-insensitive homophily** (Lim et al., 2021): for each outcome, how often its nodes link to same-outcome nodes, minus what random wiring would give. Random wiring scores about **0** at any prevalence, and perfect segregation scores **1**. That correction matters because Task A (Long Length of Stay) has a different prevalence (0.25).

**Dirichlet energy** asks the same question of the CLMBR fingerprints: how different a node's fingerprint is from its neighbors', normalized across graphs. **Lower means neighbors already look alike to CLMBR.** It needs CLMBR fingerprints, so it appears only when a full-release copy is on this machine; otherwise its row in the table below is empty.

Both use every node's label or fingerprint. They describe the graph, and must never be used to choose a model.

In [ ]:
summary_table(
    an.rows,
    [
        "edge_homophily",
        "class_insensitive_homophily",
        "same_outcome_rate_positive",
        "same_outcome_rate_negative",
        "dirichlet_energy_clmbr",
    ],
)

In [ ]:
fig = nf.homophily_energy_figure(an)
nf.save(fig, figures_dir / f"{FIG}_homophily_dirichlet.png")
plt.show()

---
# Part 4 — What does leakage actually change? (issue #15)

Three checks:

1. **Same-patient links.** A patient with several labeled admissions appears as several nodes, and those nodes tend to link to each other: their histories overlap, and in the leaky graph they are *identical*, since every admission sees the whole record. Those links share an outcome far more often than chance. So homophily is reported both with and without them. The official splits are per patient, so these links never cross train and test.
2. **Which links leakage swaps.** Links in both graphs, only in safe, and only in leaky. If leakage helps, the leaky-only links should join same-outcome nodes more often than the safe-only links they replaced.
3. **Single-code leakage.** For each diagnosis code, the share of positive vs. negative nodes whose vector contains it. In the leaky vectors a huge gap means the future record contains the answer itself.

In [ ]:
summary_table(
    an.rows,
    [
        "same_patient_edge_fraction",
        "same_patient_edge_same_outcome_rate",
        "class_insensitive_homophily",
        "class_insensitive_homophily_excl_same_patient",
        "max_single_code_outcome_gap",
    ],
)

In [ ]:
edge_sets = pd.DataFrame(
    {k: an.edge_sets[k] for k in ("shared_edges", "safe_only_edges", "leaky_only_edges")}
).T
print(f"edge Jaccard (safe vs leaky): {an.edge_sets['edge_jaccard']:.3f}")
edge_sets

In [ ]:
# The ten codes whose presence in the LEAKY vectors best separates positives
# from negatives -- code names and group rates only, no patient-level data.
names = concept_names(an.top_leaky_codes["code"], sources.concept_table)
an.top_leaky_codes.assign(name=an.top_leaky_codes["code"].map(names)).round(3)

In [ ]:
fig = nf.leakage_edges_figure(tg, an)
nf.save(fig, figures_dir / f"{FIG}_leakage_edges.png")
plt.show()

---
## Save the aggregate results

`write_aggregate` checks the schema and refuses patient-identifying columns before anything touches disk.

In [ ]:
out = write_aggregate(an.rows, results_dir / "network_stats_task_b.csv")
saved = pd.read_csv(out)
print(f"wrote {len(saved)} rows -> {out.relative_to(_repo_root)}")
print(sorted(p.name for p in figures_dir.glob(f"{FIG}_*.png")))
saved.groupby("condition")["metric"].count()

## Findings

Numbers from the Oct 2, 2026 run on the course OMOP tables: 3,511 discharges from 2,217 patients, prevalence 0.128. CLMBR fingerprints were available for 3,500 (99.7%). "Null" means the mean of 10 rewired graphs, with the middle-95% range in brackets.

**1. The same structural picture as Task A.** Transitivity is 0.479 (safe) and 0.589 (leaky), against 0.002 for null. The average path in the largest piece is 8.1 and 9.4 hops, against 4.8. Modularity is 0.92 / 0.96, against 0.54. Louvain and Leiden agree (ARI 0.77 / 0.76, against 0.25). Centrality carries no outcome information (AUROC 0.47–0.58).

**2. Here, leakage visibly improves outcome alignment.**

| | safe | leaky | null |
|---|---|---|---|
| class-insensitive homophily | 0.226 | **0.263** | 0.002 [0.000–0.009] |
| same, without same-patient links | 0.115 | **0.154** | 0.002 |
| outcome variance explained by community (η²) | 0.074 | **0.136** | 0.010 [0.005–0.015] |
| Dirichlet energy (CLMBR) | 0.653 | 0.657 | 0.945 [0.943–0.946] |

Leaky homophily is 16% higher than safe, and **34% higher once same-patient links are removed**. Communities explain nearly twice as much of the outcome. Task A shows no such effect.

**3. The future record contains the answer.** In the leaky vectors the essential-hypertension concept appears in **99.6% of positives against 12.8% of negatives**: a positive patient, by definition, gets the diagnosis within a year. The strongest single-code gap is **0.87 for leaky against 0.11 for safe**.

**4. The graph dilutes even a near-perfect answer key.** A concept present in almost every positive moves homophily by only +0.04. It is one of about 100 concepts per vector, and TF-IDF down-weights common concepts, which this is. How much leakage reaches a graph depends on how the graph is built.

**5. Same-patient links carry half of Task B's homophily.** They are 29.5% of links and agree on the outcome 94.9% of the time (chance: 77.6%). Removing them cuts safe homophily from 0.226 to 0.115. The official splits are per patient, so this never crosses train and test. But much of what a graph model could "borrow" in Task B is the same patient again.

**6. The links leakage adds are only slightly better.** Leaky-only links join same-outcome nodes 77.9% of the time; the safe-only links they replace, 76.2% (chance: 77.6%). The leaky graph's advantage is spread thinly across many neighborhoods.

### What this means for the project

- **Task B is where the paper's leakage warning should show.** Expect a larger leaky-vs-safe gap in Project 2's predictions here than for long stay.
- **Same-patient links need an explicit decision before Project 2.** Report both versions, and include a "no same-patient links" graph variant in the sweep.

### Caveats

- Same as notebook 07.
- These labels follow EHRSHOT's written rule and skip discharges where the patient already has the diagnosis. Stanford's released labels include some such patients. Notebook 11 compares the two.

### Done when
- [x] #16: the full analysis repeated for Hypertension through the same library code as Task A
- [x] #15 checks for Task B
- [x] #37: runs from the course OMOP tables alone
- [x] Only aggregate output: `results/aggregate/network_stats_task_b.csv` and figures `10_*.png`